# Módulo 3 — Aterrizaje Propulsado (Landing)
**Programa:** Inspira STEM 2026  
**Instructores:** Oscar Tejada y Patricia Ortíz

---
## La Fase Terminal y Control Vectorial
A 1.8 kilómetros de la superficie, la densidad atmosférica resulta insuficiente y la influencia del paracaídas concluye. El sistema se separa y entra en caída libre, confiando su supervivencia a la etapa de descenso (la revolucionaria maniobra *Sky Crane* implementada en Curiosity).

Este vehículo dependió de 8 motores aceleradores regulables de hidracina (Mars Lander Engines), generando empuje retrogrado para reducir la velocidad desde aproximadamente 100 m/s hasta apenas 0.75 m/s, manteniendo un vuelo estacionario (hover) milimétrico para descender el rover mediante un umbilical de nylon.

La viabilidad termodinámica y orbital de esta maniobra se consolida evaluando tres balances paramétricos fundamentales.

---

### Paso 1: Configuración de la Computadora de Vuelo
Ejecuta la celda inferior para compilar el módulo de empuje de la computadora de a bordo. Recuerda registrar con precisión la gravedad local y el requisito de cambio de velocidad ($\Delta v$) heredado de la pérdida de eficiencia del paracaídas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

plt.rcParams.update({'figure.dpi': 100, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 2})

# ==========================================
# ⚙️ DATOS DEL ENTORNO Y REQUISITO CINEMÁTICO
# ==========================================
g_planeta = 3.71        # Gravedad local [m/s^2]
dv_requerido = 800.0    # Velocidad residual a anular [m/s]

def calc_landing(m_seca, m_prop, isp, empuje):
    m_tot = m_seca + m_prop
    return {
        "peso_kn": (m_tot * g_planeta) / 1000,
        "empuje_kn": empuje / 1000,
        "twr": empuje / (m_tot * g_planeta),
        "dv_max": isp * 9.81 * np.log(m_tot / m_seca),
        "tb": (m_prop * isp * 9.81) / empuje
    }
print("Sistemas de retropropulsión en línea.")

---
### Análisis 1: Relación Empuje-Peso (TWR)
La autoridad de control vertical absoluta está dictaminada por la relación *Thrust-to-Weight Ratio* (TWR). Este indicador contrasta el **Empuje total ($\mathbf{F}$)** generado por el bloque motriz frente a la atracción gravitatoria sobre la masa del sistema:
$$TWR = \frac{\mathbf{F}}{m_{total} \cdot g_{local}}$$

La exploración de este parámetro, manipulando el Empuje ($\mathbf{F}$), permite comprobar visualmente que la maniobra de frenado requiere estrictamente que el vector de fuerza ascendente (empuje operativo) supere el componente vectorial de descenso (peso local), logrando un $TWR > 1$.

In [ ]:
def interact_empuje(empuje_kn):
    d = calc_landing(m_seca=800.0, m_prop=400.0, isp=300.0, empuje=empuje_kn*1000)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    gravedades = np.linspace(1.0, 10.0, 50)
    twrs = (empuje_kn * 1000) / (1200 * gravedades)
    axs[0].plot(gravedades, twrs, color="#c0392b")
    axs[0].axhline(1.0, color="k", linestyle="--", label="Límite Operacional Crítico (TWR = 1)")
    axs[0].set(title="Comportamiento TWR frente a Gravedades Planetarias", xlabel="Gravedad Local [m/s²]", ylabel="Relación Empuje/Peso")
    axs[0].legend()
    
    axs[1].bar(["Peso Sistémico"], [d["peso_kn"]], color="#c0392b")
    axs[1].bar(["Empuje de Reacción"], [d["empuje_kn"]], color="#27ae60")
    axs[1].set(title=f"Balance de Fuerzas Verticales (TWR Actual: {d['twr']:.2f})", ylabel="Fuerza Neta [kN]")
    plt.show()

interact(interact_empuje, empuje_kn=widgets.FloatSlider(value=25.0, min=5.0, max=60.0, step=1.0, description='Empuje [kN]:'));

---
### Análisis 2: Presupuesto Cinemático y Ecuación de Tsiolkovsky
El monto total de cambio de velocidad ($\,\Delta v\,$) que la configuración propulsiva puede brindar se rige incondicionalmente por la Ecuación del Cohete de Tsiolkovsky.

Esta ley física subraya el impacto crítico del rendimiento termodinámico de la ignición, cuantificado a través del **Impulso Específico ($\mathbf{I_{sp}}$)**:
$$\Delta v = \mathbf{I_{sp}} \cdot g_0 \cdot \ln\left(\frac{m_{total}}{m_{seca}}\right) \quad [\text{m/s}]$$

Iterando sobre el Impulso Específico ($I_{sp}$), se simula la transición entre distintas arquitecturas químicas de propelentes (ej. monopropelentes de hidracina frente a sistemas bipropelentes más complejos) para garantizar que la capacidad $\Delta v$ de la plataforma cumpla con el requerimiento de la misión.

In [ ]:
def interact_isp(isp_segundos):
    d = calc_landing(m_seca=800.0, m_prop=400.0, isp=isp_segundos, empuje=25000.0)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    fracciones = np.linspace(1.1, 2.5, 50)
    dvs = isp_segundos * 9.81 * np.log(fracciones)
    axs[0].plot(fracciones, dvs, color="#2980b9")
    axs[0].set(title="Curva Potencial de Tsiolkovsky", xlabel="Fracción de Masa (Húmeda / Seca)", ylabel="$\Delta v$ Alcanzable [m/s]")
    
    axs[1].barh(["Capacidad de la Etapa"], [d["dv_max"]], color="#2980b9")
    axs[1].axvline(dv_requerido, color="#e67e22", linestyle="--", lw=3, label=f"Requisito Mínimo ({dv_requerido} m/s)")
    axs[1].set(title="Presupuesto Cinemático Consolidado", xlabel="Cambio de Velocidad $\Delta v$ [m/s]")
    axs[1].legend()
    plt.show()

interact(interact_isp, isp_segundos=widgets.FloatSlider(value=300.0, min=150.0, max=450.0, step=10.0, description='Eficiencia (Isp):'));

---
### Análisis 3: Restricción Temporal de Vuelo Estacionario
La maniobra crítica de separación y descenso del rover con poleas y cables demanda mantener un estado de *hover* preciso (aceleración neta de 0 m/s²) durante los últimos segundos. Esta ventana de ignición prolongada está coartada por el volumen de **Masa del Propelente ($\mathbf{m_p}$)** en los depósitos de hidracina:
$$t_b = \frac{\mathbf{m_p} \cdot I_{sp} \cdot g_0}{F} \quad [\text{s}]$$

La evaluación de la capacidad de los tanques ($m_p$) revela la fragilidad temporal de la etapa terminal, evidenciando la necesidad de conservar márgenes de fluido operativo hasta el instante del contacto superficial.

In [ ]:
def interact_tanque(masa_tanque):
    d = calc_landing(m_seca=800.0, m_prop=masa_tanque, isp=300.0, empuje=25000.0)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    empujes = np.linspace(10, 50, 50)
    t_burns = (masa_tanque * 300.0 * 9.81) / (empujes * 1000)
    axs[0].plot(empujes, t_burns, color="#8e44ad")
    axs[0].set(title="Tasa de Agotamiento vs Fuerza Propulsiva", xlabel="Empuje Operativo [kN]", ylabel="Duración Máxima [s]")
    
    axs[1].bar(["Margen Restante"], [d["tb"]], color="#8e44ad")
    axs[1].axhline(30.0, color="#2c3e50", linestyle="--", lw=2, label="Ventana de Descenso (30s)")
    axs[1].scatter(["Margen Restante"], [47.0], color="gold", marker="D", s=100, edgecolor="k", zorder=5, label="Margen Histórico Curiosity")
    axs[1].set(title="Límite Temporal del Aterrizaje", ylabel="Tiempo [s]")
    axs[1].legend()
    plt.show()

interact(interact_tanque, masa_tanque=widgets.FloatSlider(value=400.0, min=100.0, max=1000.0, step=50.0, description='Propelente [kg]:'));

### Análisis de Grupo
Reúnanse y debatan las exigencias y dependencias paramétricas del aterrizaje propulsado:

1. **Configuración de Motores (Misión Real):** El vehículo de descenso de Curiosity utilizó 8 motores aceleradores MLE (Mars Lander Engines) distribuidos alrededor de la cápsula. Si por un exceso de "seguridad" los ingenieros hubieran ordenado que operaran a su empuje máximo estructural para frenar lo antes posible, ¿qué crisis inmediata se habría detonado según la matemática del Análisis 3?
2. **El Reloj de la Grúa Aérea:** La maniobra de *Sky Crane* exigía que la etapa de descenso sostuviera el peso colosal del rover Curiosity, volando estacionariamente, durante aproximadamente 21 a 25 segundos críticos. Si su simulación arroja un agotamiento de combustible prematuro a los 18 segundos, ¿qué dos variables cruzadas del sistema podrían alterarse para recuperar ese margen de operación sin penalizar fatalmente a la nave por exceso de peso?
3. **El Escape de la Grúa (Misión Real):** En agosto de 2012, una vez que las ruedas del Curiosity tocaron el cráter Gale y los umbilicales pirotécnicos se cortaron, los motores de la grúa no se apagaron. Su sistema de vuelo ordenó una maniobra de *Flyaway* (Escape), acelerando a máxima potencia hacia arriba e inclinándose para alejarse del rover e impactar intencionalmente a 650 metros de distancia. Observando sus gráficas del Análisis 2 y 3, ¿qué requerimiento exigía esta maniobra de escape obligatorio al diseño original de los depósitos de hidracina, y por qué la NASA nunca dimensiona el combustible para llegar exactamente a cero en el instante de contacto?